# 01 — Análise exploratória

**CRISP-DM fase 2.3 — exploração dos dados** · Meridiano Imobiliária

---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de Ames, usando o dicionário de dados montado em
[`0-dicionario-dados.ipynb`](0-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


In [1]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import circlify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import prince
import seaborn as sns

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.data import ingestao
from src.eda import estatistica as est
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

io.configurar_estilo()

TITULO = "SEGMENTAÇÃO DA CARTEIRA RESIDENCIAL"

# As figuras deste roteiro vão para `docs/imagens/eda/`, separadas das evidências
# canônicas do pipeline (`docs/imagens/evidencias/eda/`, gravadas por
# `src.eda.fluxo8fases`). São duas leituras da mesma base e não devem se misturar.
FIGURAS = config.DOCS / "imagens" / "eda"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "eda-exploratoria"

# Cores da identidade do projeto (docs/assets/css/brand.css), no lugar do azul e do
# amarelo fixos do script de referência.
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO

#: Quantas categorias entram nos gráficos de frequência. Bairro tem 28 níveis;
#: mostrar todos transforma o gráfico em lista.
TOPO = 10


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/eda/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


print(f"Projeto em {RAIZ}")

Projeto em d:\repositorios\UNIFOR\unifor-disciplina-aprendizado-nao-supervisionado\projeto-exemplo-mercado-imobiliario


## Carregando a base de dados

Duas entradas: o **dicionário de dados** (`data/processed/dicionario.csv`, produzido
pelo notebook `0-dicionario-dados`) e a **base bruta** (`data/raw/ames_housing.parquet`), que a ingestão já grava com
nomes e categorias em português.
O dicionário manda: as colunas selecionadas são exatamente as que ele descreve, e o
campo `Tipo` é o que decide qual análise cada característica recebe.

In [2]:
# Carregando o dicionário de dados
dicionario = pd.read_csv(config.DATA_PROCESSED / "dicionario.csv", sep=";")

# Carregando base de dados — a camada bruta em português. `carregar_bruto` a
# materializa a partir do .txt da fonte se a ingestão ainda não tiver rodado.
bruto = ingestao.carregar_bruto()

# Selecionando as mesmas colunas do dicionário de dados
bruto = bruto.loc[:, dicionario.variavel]

print(f"{bruto.shape[0]:,} linhas × {bruto.shape[1]} colunas")
bruto.head()

2,930 linhas × 82 colunas


,ordem,pid,subclasse,zoneamento,testada_lote,area_terreno,tipo_via,tipo_viela,formato_lote,relevo,infraestrutura,config_lote,declividade,bairro,condicao_entorno_1,condicao_entorno_2,tipo_construcao,estilo,qualidade_geral,conservacao_geral,...,vagas_garagem,area_garagem,qualidade_garagem,conservacao_garagem,acesso_pavimentado,area_deck,area_varanda_aberta,area_varanda_fechada,area_varanda_3estacoes,area_varanda_tela,area_piscina,qualidade_piscina,tipo_cerca,item_extra,valor_item_extra,mes_venda,ano_venda,tipo_venda,condicao_venda,preco_venda
0,1,526301100,20,Residencial de baixa densidade,141.0,31770,Pavimentada,NaN,Levemente irregular,Plano,Todos os serviços,Esquina,Suave,North Ames,Normal,Normal,Unifamiliar isolada,Térrea,6,5,...,2.0,528.0,Típico,Típico,Parcialmente pavimentado,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,Escritura convencional,Normal,215000
1,2,526350040,20,Residencial de alta densidade,80.0,11622,Pavimentada,NaN,Regular,Plano,Todos os serviços,Meio de quadra,Suave,North Ames,Junto a via coletora,Normal,Unifamiliar isolada,Térrea,5,6,...,1.0,730.0,Típico,Típico,Pavimentado,140,0,0,0,120,0,NaN,Privacidade mínima,NaN,0,6,2010,Escritura convencional,Normal,105000
2,3,526351010,20,Residencial de baixa densidade,81.0,14267,Pavimentada,NaN,Levemente irregular,Plano,Todos os serviços,Esquina,Suave,North Ames,Normal,Normal,Unifamiliar isolada,Térrea,6,6,...,1.0,312.0,Típico,Típico,Pavimentado,393,36,0,0,0,0,NaN,NaN,Segunda garagem,12500,6,2010,Escritura convencional,Normal,172000
3,4,526353030,20,Residencial de baixa densidade,93.0,11160,Pavimentada,NaN,Regular,Plano,Todos os serviços,Esquina,Suave,North Ames,Normal,Normal,Unifamiliar isolada,Térrea,7,5,...,2.0,522.0,Típico,Típico,Pavimentado,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,Escritura convencional,Normal,244000
4,5,527105010,60,Residencial de baixa densidade,74.0,13830,Pavimentada,NaN,Levemente irregular,Plano,Todos os serviços,Meio de quadra,Suave,Gilbert,Normal,Normal,Unifamiliar isolada,Dois pavimentos,5,5,...,2.0,482.0,Típico,Típico,Pavimentado,212,34,0,0,0,0,NaN,Privacidade mínima,NaN,0,3,2010,Escritura convencional,Normal,189900


## O recorte da análise

O script de referência analisa **todas** as colunas do dicionário e cruza todas contra
todas. Com 82 características isso dá 259 figuras univariadas e mais de 1.500 bivariadas
— volume que ninguém audita e que não cabe em documentação.

O recorte aqui **não é arbitrário**: são as variáveis que compõem o espaço de atributos
declarado em `src/data/dicionario.py` (`NOMINAIS`, `ORDINAIS`, `NUMERICAS` e
`ATRIBUTOS_TRANSACAO`) — com o mesmo nome na base bruta, que a ingestão já grava em
português. Entram também as duas **reservadas** — `bairro`, guardada para validação
externa, e `preco_venda`, o rótulo que só é usado na avaliação a posteriori — e
`testada_lote`, a única contínua da base com ausência que **não** é estrutural.

As quatro numéricas derivadas do espaço curado (`area_varandas`, `banheiros_total`,
`idade_imovel`, `anos_desde_reforma`) não existem na base bruta e por isso ficam de
fora: este notebook descreve o dado como ele chega, antes da preparação.

`mes_venda` e `ano_venda` saem da lista depois de recompostas em `data_venda` — as duas
juntas são uma data, e é como data que a 2ª e a 3ª etapas as usam.

In [3]:
ESPACO = (
    dic.NOMINAIS
    + dic.ORDINAIS
    + dic.NUMERICAS
    + dic.ATRIBUTOS_TRANSACAO
    + [dic.RESERVADA_VALIDACAO, dic.ROTULO_RESERVADO]
)

# As derivadas do espaço (`area_varandas`, `idade_imovel`…) ainda não existem na base
# bruta; o recorte fica com as que existem, mais a testada do lote.
RECORTE = [coluna for coluna in ESPACO if coluna in bruto.columns] + ["testada_lote"]

df = bruto.loc[:, RECORTE].copy()

# Recompondo o tempo: a base guarda ano e mês em colunas separadas.
df["data_venda"] = pd.to_datetime(dict(year=df["ano_venda"], month=df["mes_venda"], day=1))
df = df.drop(columns=["mes_venda", "ano_venda"])

# O dicionário acompanha o recorte, e a coluna derivada entra nele com o tipo `Data`.
dicionario = dicionario[dicionario.variavel.isin(df.columns)].reset_index(drop=True)
dicionario.loc[len(dicionario)] = {
    "variavel": "data_venda",
    "coluna_original": "Yr Sold + Mo Sold",
    "descricao": "Mês de referência da venda, recomposto de ano_venda e mes_venda",
    "valores": [str(df["data_venda"].min().date()), str(df["data_venda"].max().date())],
    "Tipo": "Data",
}

print(f"{df.shape[1]} características analisadas, de {bruto.shape[1]} descritas")
dicionario.Tipo.value_counts()

24 características analisadas, de 82 descritas


Tipo
Nominal     8
Discreta    8
Ordinal     4
Contínua    3
Data        1
Name: count, dtype: int64

Com o recorte fechado, cada tipo declarado no dicionário vira uma lista — é essa lista
que governa quais gráficos cada característica recebe nas etapas seguintes.

In [4]:
def por_tipo(tipo: str) -> list[str]:
    """Nomes das colunas de um dado tipo semântico, na ordem do dicionário."""
    return dicionario.query("Tipo == @tipo").variavel.tolist()


nominal_columns = por_tipo("Nominal")
ordinal_columns = por_tipo("Ordinal")
continuous_columns = por_tipo("Contínua")
discrete_columns = por_tipo("Discreta")
data_columns = por_tipo("Data")

# Contínuas e discretas recebem o mesmo tratamento estatístico: a distinção do
# dicionário é semântica, não computacional.
numericas = continuous_columns + discrete_columns

for nome, lista in [
    ("Nominais", nominal_columns),
    ("Ordinais", ordinal_columns),
    ("Contínuas", continuous_columns),
    ("Discretas", discrete_columns),
    ("Tempo", data_columns),
]:
    print(f"{nome:>10} ({len(lista):>2}): {', '.join(lista)}")

  Nominais ( 8): zoneamento, bairro, tipo_construcao, estilo, ar_central, tipo_garagem, tipo_venda, condicao_venda
  Ordinais ( 4): qualidade_externa, qualidade_porao, qualidade_cozinha, acabamento_garagem
 Contínuas ( 3): testada_lote, area_porao, area_garagem
 Discretas ( 8): area_terreno, qualidade_geral, conservacao_geral, area_habitavel, dormitorios, comodos, lareiras, preco_venda
     Tempo ( 1): data_venda


# 1ª etapa · Distribuição dos atributos

> *Na distribuição dos atributos é possível identificar o comportamento de cada uma das
> características analisadas.*

O script de referência obtém este panorama do `ProfileReport` do `ydata-profiling` e lê
as chaves de `profile.description_set`. A biblioteca exige `pandas < 3` e não instala
neste projeto — o panorama abaixo reproduz as mesmas contagens (`n`, `n_var`, tipos,
duplicados, células ausentes) direto do pandas.

In [5]:
def perfilar(dados: pd.DataFrame) -> dict:
    """Contagens de topo do conjunto — o que o `ProfileReport` traz em `table`."""
    quantitativas = dados.select_dtypes(include="number").columns
    celulas = dados.shape[0] * dados.shape[1]
    vazias = int(dados.isna().sum().sum())
    return {
        "Registros": dados.shape[0],
        "Características": dados.shape[1],
        "Numéricas": len(quantitativas),
        "Categóricas": dados.shape[1] - len(quantitativas),
        "Registros duplicados": int(dados.duplicated().sum()),
        "Células ausentes": vazias,
        "% de células ausentes": round(100 * vazias / celulas, 2),
    }


perfil_inicial = perfilar(df)

# Guardado para a 8ª etapa, que compara o conjunto de entrada com o selecionado.
df_inicial_qnt_feature = perfil_inicial["Características"]
df_inicial_qnt_feature_Numeric = perfil_inicial["Numéricas"]
df_inicial_qnt_feature_Categorical = perfil_inicial["Categóricas"]

pd.Series(perfil_inicial).to_frame(TITULO)

,SEGMENTAÇÃO DA CARTEIRA RESIDENCIAL
Registros,2930.00
Características,24.00
Numéricas,11.00
Categóricas,13.00
Registros duplicados,2.00
Células ausentes,888.00
% de células ausentes,1.26


## Análises iniciais

Duas linhas se repetem quando `ordem` e `pid` saem de cena — os dois identificadores são
o único ponto em que elas diferem. Um imóvel descrito duas vezes com as mesmas 24
características conta duas vezes em toda média que vier depois, então saem.

In [6]:
# Removendo registros duplicados
duplicados = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

perfil_final = perfilar(df)

panorama = pd.DataFrame({"Entrada": perfil_inicial, "Após deduplicação": perfil_final})
io.salvar_tabela(panorama.reset_index(names="medida"), SECAO, "panorama", indice=False)

print(f"{duplicados} registro(s) duplicado(s) removido(s)")
panorama

2 registro(s) duplicado(s) removido(s)


,Entrada,Após deduplicação
Registros,2930.00,2928.00
Características,24.00,24.00
Numéricas,11.00,11.00
Categóricas,13.00,13.00
Registros duplicados,2.00,0.00
Células ausentes,888.00,885.00
% de células ausentes,1.26,1.26


### Perfil coluna a coluna

O segundo bloco que o `ProfileReport` entregaria: para cada característica, quantos
valores distintos, quanto falta e — nas numéricas — mínimo, mediana, máximo e a
assimetria que decide se a variável precisa de transformação na preparação.

In [7]:
def descrever(dados: pd.DataFrame, dicionario: pd.DataFrame) -> pd.DataFrame:
    """Perfil por coluna: cardinalidade, ausência e medidas de posição e forma."""
    linhas = []
    for _, meta in dicionario.iterrows():
        coluna = dados[meta.variavel]
        quantitativa = pd.api.types.is_numeric_dtype(coluna)
        linhas.append(
            {
                "variavel": meta.variavel,
                "tipo": meta.Tipo,
                "distintos": int(coluna.nunique()),
                "ausentes": int(coluna.isna().sum()),
                "%ausentes": round(100 * coluna.isna().mean(), 2),
                "minimo": coluna.min() if quantitativa else "—",
                "mediana": coluna.median() if quantitativa else "—",
                "maximo": coluna.max() if quantitativa else "—",
                "assimetria": round(coluna.skew(), 2) if quantitativa else "—",
                "curtose": round(coluna.kurtosis(), 2) if quantitativa else "—",
            }
        )
    return pd.DataFrame(linhas)


perfil_colunas = descrever(df, dicionario)
io.salvar_tabela(perfil_colunas, SECAO, "perfil-das-colunas", indice=False)
perfil_colunas

,variavel,tipo,distintos,ausentes,%ausentes,minimo,mediana,maximo,assimetria,curtose
0,zoneamento,Nominal,7,0,0.00,—,—,—,—,—
1,testada_lote,Contínua,128,490,16.73,21.0,68.0,313.0,1.5,11.22
2,area_terreno,Discreta,1960,0,0.00,1300,9436.5,215245,12.82,264.87
3,bairro,Nominal,28,0,0.00,—,—,—,—,—
4,tipo_construcao,Nominal,5,0,0.00,—,—,—,—,—
5,estilo,Nominal,8,0,0.00,—,—,—,—,—
6,qualidade_geral,Discreta,10,0,0.00,1,6.0,10,0.19,0.05
7,conservacao_geral,Discreta,9,0,0.00,1,5.0,9,0.57,1.49
8,qualidade_externa,Ordinal,4,0,0.00,—,—,—,—,—
9,qualidade_porao,Ordinal,5,79,2.70,—,—,—,—,—


# 2ª etapa · Análise univariada

> *A análise estatística univariada analisa a distribuição e a dispersão dos dados: a
> análise da distribuição de frequências e a análise das medidas de localização,
> dispersão, assimetria e curtose.*

Cada família de tipo pede um gráfico diferente. Nominal não tem ordem nem média — só
frequência. Numérica tem posição, dispersão e forma. Ordinal tem ordem, e o gráfico
precisa respeitá-la. Tempo tem sequência.

## Características nominais

Frequência das categorias mais comuns, com o valor absoluto, a fração do total e a
linha tracejada na **frequência média por categoria** — quem passa dela concentra
carteira, quem fica muito abaixo é cauda longa.

In [12]:
for column in nominal_columns:
    contagem = df[column].value_counts()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(x=contagem.values[:TOPO], y=contagem.index[:TOPO], color=BARRA, ax=ax)

    # Valores absolutos e relativos sobre cada barra
    total = float(contagem.sum())
    for grupo in ax.containers:
        larguras = [barra.get_width() for barra in grupo]
        ax.bar_label(
            grupo,
            label_type="edge",
            labels=[f"{largura:,.0f} · {largura / total:.1%}" for largura in larguras],
            fontsize=8,
            padding=3,
        )

    # Linha de referência na frequência média por categoria
    ax.axvline(contagem.values.mean(), color=REFERENCIA, ls="dashed", lw=2)

    ax.set_xlabel("Quantidade de registros")
    ax.set_ylabel(column)
    ax.set_title(f"Categorias mais frequentes de {column} ({contagem.size} no total)")
    ax.margins(x=0.15)

    salvar(fig, f"univariada-nominal-{column}", contagem.rename("registros").reset_index())

print(f"{len(nominal_columns)} figuras nominais gravadas")

8 figuras nominais gravadas


## Características contínuas e discretas

Os quatro gráficos que o script de referência grava em arquivos separados — extremos e
amplitude, distribuição, dispersão registro a registro e diagrama de caixa — vão para um
painel único por característica. O histograma entra no lugar do segundo diagrama de
caixa (o dos quartis), que repetia a informação da caixa ao lado sem acrescentar nada:
é a distribuição, e não os quartis outra vez, que revela cauda e concentração.

O título de cada painel traz a **assimetria**, a medida que decide na fase de preparação
quem recebe `log1p` — o projeto usa 0,75 como limiar (`dic.LIMIAR_ASSIMETRIA`).

In [13]:
def painel_numerico(column: str, familia: str) -> tuple[plt.Figure, pd.DataFrame]:
    """Extremos, distribuição, dispersão e caixa de uma característica numérica."""
    serie = df[column].dropna()
    q1, q2, q3 = serie.quantile([0.25, 0.5, 0.75])

    fig, eixos = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"{column} — {familia} · assimetria {serie.skew():.2f}",
        fontsize=13,
        fontweight="bold",
        color=config.AZUL_PROFUNDO,
    )

    # Valores mínimo e máximo, com a amplitude na linha tracejada
    ax = eixos[0, 0]
    sns.barplot(x=["Mínimo", "Máximo"], y=[serie.min(), serie.max()], color=BARRA, ax=ax)
    for grupo in ax.containers:
        ax.bar_label(grupo, fmt="%.0f", fontsize=9, padding=3)
    ax.axhline(serie.max() - serie.min(), color=REFERENCIA, ls="dashed", lw=2)
    ax.set_title("Extremos e amplitude")
    ax.set_ylabel(column)

    # Distribuição de frequências, com a média na linha tracejada
    ax = eixos[0, 1]
    sns.histplot(serie, bins=30, color=BARRA, ax=ax)
    ax.axvline(serie.mean(), color=REFERENCIA, ls="dashed", lw=2)
    ax.set_title(f"Distribuição · média {serie.mean():,.0f} · mediana {q2:,.0f}")
    ax.set_xlabel(column)

    # Dispersão registro a registro — mostra concentração e pontos soltos
    ax = eixos[1, 0]
    sns.scatterplot(x=serie.index, y=serie.values, color=BARRA, s=12, alpha=0.5, ax=ax)
    ax.set_title("Dispersão por registro")
    ax.set_xlabel("Índice do registro")
    ax.set_ylabel(column)

    # Diagrama de caixa, com a mediana destacada
    ax = eixos[1, 1]
    sns.boxplot(
        x=serie,
        color=BARRA,
        width=0.45,
        medianprops={"color": REFERENCIA, "linewidth": 2},
        ax=ax,
    )
    ax.set_title(f"Caixa · Q1 {q1:,.0f} · Q2 {q2:,.0f} · Q3 {q3:,.0f}")
    ax.set_xlabel(column)

    fig.tight_layout()

    resumo = pd.DataFrame(
        [
            {
                "variavel": column,
                "minimo": serie.min(),
                "q1": q1,
                "mediana": q2,
                "q3": q3,
                "maximo": serie.max(),
                "media": serie.mean(),
                "desvio": serie.std(),
                "assimetria": serie.skew(),
                "curtose": serie.kurtosis(),
            }
        ]
    )
    return fig, resumo


resumos = []
for column in continuous_columns:
    fig, resumo = painel_numerico(column, "característica contínua")
    salvar(fig, f"univariada-continua-{column}", resumo)
    resumos.append(resumo)

for column in discrete_columns:
    fig, resumo = painel_numerico(column, "característica discreta")
    salvar(fig, f"univariada-discreta-{column}", resumo)
    resumos.append(resumo)

resumo_numericas = pd.concat(resumos, ignore_index=True).sort_values(
    "assimetria", ascending=False
)
io.salvar_tabela(resumo_numericas, SECAO, "univariada-resumo-numericas", indice=False)
resumo_numericas

,variavel,minimo,q1,mediana,q3,maximo,media,desvio,assimetria,curtose
3,area_terreno,1300.0,7440.75,9436.5,11556.0,215245.0,10148.768101,7882.487925,12.817306,264.865622
10,preco_venda,12789.0,129500.00,160000.0,213500.0,755000.0,180817.827186,79905.769953,1.742784,5.114674
0,testada_lote,21.0,58.00,68.0,80.0,313.0,69.230517,23.373933,1.497912,11.224071
6,area_habitavel,334.0,1126.00,1442.0,1743.0,5642.0,1499.780738,505.650793,1.273443,4.133754
1,area_porao,0.0,793.00,990.0,1302.0,6110.0,1051.923129,440.328019,1.161801,9.161220
8,comodos,2.0,5.00,6.0,7.0,15.0,6.442964,1.573012,0.753995,1.156429
9,lareiras,0.0,0.00,1.0,1.0,4.0,0.599727,0.647953,0.738256,0.100650
5,conservacao_geral,1.0,5.00,5.0,6.0,9.0,5.563525,1.111819,0.573435,1.489120
7,dormitorios,0.0,2.00,3.0,3.0,8.0,2.853825,0.827739,0.306586,1.893991
2,area_garagem,0.0,320.00,480.0,576.0,1488.0,473.006150,214.938076,0.243577,0.952848


## Características ordinais

O ramo ordinal do script de referência nunca chega a rodar — nenhuma coluna recebe esse
tipo na varredura original. Aqui ele existe porque o dicionário do projeto declara as
escalas em `ESCALAS_ORDINAIS`, e é essa declaração que ordena as barras: a frequência
sai na ordem da escala (`Ruim < Regular < Típico < Bom < Excelente`), não na ordem de frequência.

Ausência em característica ordinal com **ausência estrutural** não é dado faltante: `NA`
em `qualidade_porao` significa "o imóvel não tem porão". Por isso o nível `Ausente` aparece no
gráfico de frequência e vale 0 na escala codificada — fora da escala, abaixo do pior
nível existente.

Ao lado, a frequência acumulada na ordem da escala: onde a linha cruza os 50% está o
nível mediano da carteira. É a leitura de dispersão que faz sentido em variável ordinal
— quartil e desvio padrão pressupõem distância entre níveis, e `Bom` não fica a "uma
unidade" de `Típico`.

In [14]:
for column in ordinal_columns:
    escala = dic.ESCALAS_ORDINAIS[column]
    ordem = sorted(escala, key=escala.get)

    contagem = df[column].fillna("Ausente").value_counts()
    presentes = [nivel for nivel in ordem if nivel in contagem.index]
    contagem = contagem.reindex(presentes + [n for n in contagem.index if n not in ordem])

    fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"{column} — característica ordinal",
        fontsize=13,
        fontweight="bold",
        color=config.AZUL_PROFUNDO,
    )

    # Frequência na ordem da escala, não na ordem de frequência
    ax = eixos[0]
    sns.barplot(x=contagem.index, y=contagem.values, color=BARRA, ax=ax)
    for grupo in ax.containers:
        ax.bar_label(grupo, fmt="%.0f", fontsize=9, padding=3)
    ax.set_title("Frequência por nível, na ordem da escala")
    ax.set_ylabel("Registros")
    ax.set_xlabel(column)

    # Frequência acumulada na ordem da escala: onde está a metade da carteira
    ax = eixos[1]
    acumulada = 100 * contagem.reindex(presentes).cumsum() / contagem.sum()
    sns.lineplot(x=acumulada.index, y=acumulada.values, color=BARRA, marker="o", lw=2, ax=ax)
    ax.axhline(50, color=REFERENCIA, ls="dashed", lw=2)
    codificada = df[column].map(escala).fillna(0)
    ax.set_ylim(0, 105)
    ax.set_ylabel("% acumulado")
    ax.set_title(f"Frequência acumulada · nível mediano {codificada.median():.0f} na escala")
    ax.set_xlabel(" < ".join(ordem))

    fig.tight_layout()
    salvar(fig, f"univariada-ordinal-{column}", contagem.rename("registros").reset_index())

print(f"{len(ordinal_columns)} figuras ordinais gravadas")

4 figuras ordinais gravadas


## Características de tempo

A base não traz coluna de data; `data_venda` é a recomposição de `ano_venda` com `mes_venda`
feita no recorte. A linha do tempo mede **volume de transações por mês**, com a média
mensal na linha tracejada.

In [15]:
for column in data_columns:
    serie = df[column].value_counts().sort_index().reset_index()
    serie.columns = ["data", "qnt"]

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.lineplot(data=serie, x="data", y="qnt", color=BARRA, lw=2.5, ax=ax)
    ax.axhline(serie["qnt"].mean(), color=REFERENCIA, ls="dashed", lw=2)
    ax.set_ylabel("Quantidade de registros")
    ax.set_xlabel(column)
    ax.set_title(f"Distribuição na linha do tempo de {column}")

    salvar(fig, f"univariada-tempo-{column}", serie)

serie

,data,qnt
0,2006-01-01,18
1,2006-02-01,24
2,2006-03-01,51
3,2006-04-01,48
4,2006-05-01,75
5,2006-06-01,97
6,2006-07-01,122
7,2006-08-01,45
8,2006-09-01,41
9,2006-10-01,49


# 3ª etapa · Análise bivariada

> *Na avaliação estatística da causalidade, convencionou-se descrever essas análises
> bivariadas como formas de investigar se a variação de uma característica pode ser
> explicada pela variação de outra medida.*

O script cruza cada nominal contra cada numérica. Com o recorte deste notebook seriam
88 figuras; o cruzamento aqui parte de **quatro eixos de leitura** — os recortes pelos
quais a Meridiano organiza a carteira — contra **cinco medidas** de tamanho, qualidade e
preço. Cada barra traz a média do grupo e quanto ela representa da média geral, marcada
na linha tracejada.

## Nominais × contínuas e discretas

In [16]:
#: Os recortes pelos quais o negócio lê a carteira.
EIXOS = ["bairro", "tipo_construcao", "zoneamento", "condicao_venda"]

#: As medidas que o Canvas do Problema nomeia: tamanho, qualidade e preço.
MEDIDAS_CONTINUAS = ["area_porao", "area_garagem"]
MEDIDAS_DISCRETAS = ["preco_venda", "area_habitavel", "qualidade_geral"]


def bivariada_barras(categoria: str, medida: str, familia: str) -> pd.DataFrame:
    """Média da medida por categoria, ordenada, com a média geral como referência."""
    tabela = (
        df.groupby(categoria, observed=True)[medida]
        .agg(media="mean", registros="size")
        .reset_index()
        .sort_values("media", ascending=False)
    )
    recorte = tabela.head(TOPO)
    media_geral = df[medida].mean()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(x=recorte["media"], y=recorte[categoria], color=BARRA, ax=ax)
    for grupo in ax.containers:
        larguras = [barra.get_width() for barra in grupo]
        ax.bar_label(
            grupo,
            label_type="edge",
            labels=[f"{v:,.0f} · {v / media_geral:.0%}" for v in larguras],
            fontsize=8,
            padding=3,
        )
    ax.axvline(media_geral, color=REFERENCIA, ls="dashed", lw=2)
    ax.set_xlabel(f"Média de {medida}")
    ax.set_ylabel(categoria)
    ax.set_title(f"{categoria} × média de {medida} — característica {familia}")
    ax.margins(x=0.18)

    salvar(fig, f"bivariada-{medida} X {categoria}", tabela)
    return tabela


for categoria in EIXOS:
    for medida in MEDIDAS_CONTINUAS:
        bivariada_barras(categoria, medida, "contínua")
    for medida in MEDIDAS_DISCRETAS:
        bivariada_barras(categoria, medida, "discreta")

print(f"{len(EIXOS) * (len(MEDIDAS_CONTINUAS) + len(MEDIDAS_DISCRETAS))} figuras gravadas")

20 figuras gravadas


## Nominais × tempo

O mesmo cruzamento, agora contra a linha do tempo: as seis categorias mais frequentes de
cada eixo, mês a mês. O que se procura aqui é **mudança de composição** — categoria que
cresce ou desaparece ao longo dos cinco anos da série.

In [17]:
for tempo in data_columns:
    for nominal in EIXOS:
        # Só as categorias mais frequentes: as demais viram ruído de legenda.
        frequentes = df[nominal].value_counts().head(6).index

        serie = (
            df[df[nominal].isin(frequentes)]
            .groupby([tempo, nominal], observed=True)
            .size()
            .reset_index(name="Contagem")
        )

        fig, ax = plt.subplots(figsize=(16, 5))
        sns.lineplot(
            data=serie,
            x=tempo,
            y="Contagem",
            hue=nominal,
            palette=config.PALETA[: len(frequentes)],
            lw=2,
            ax=ax,
        )
        ax.axhline(serie["Contagem"].mean(), color=REFERENCIA, ls="dashed", lw=2)
        ax.set_ylabel("Quantidade de registros")
        ax.set_xlabel(tempo)
        ax.set_title(f"{tempo} × {nominal} — as seis categorias mais frequentes")
        ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", title=nominal)

        salvar(fig, f"bivariada-{nominal} X {tempo}", serie)

print(f"{len(EIXOS) * len(data_columns)} figuras de linha do tempo gravadas")

4 figuras de linha do tempo gravadas


# 4ª etapa · Análise multivariada

> *Análise multivariada é uma ferramenta que encontra padrões e relações entre várias
> variáveis, permitindo prever efeitos e mudanças que uma variável terá sobre a outra.*

Duas variáveis nominais não têm correlação de Pearson — não têm ordem para correlacionar.
A **análise de correspondência** (`prince.CA`) resolve isso: decompõe a tabela de
contingência e projeta linhas e colunas no mesmo plano, onde a proximidade entre um
bairro e um tipo de garagem significa que aquela combinação aparece mais do que o acaso
explicaria.

Cada categoria de linha é então vinculada à categoria de coluna **mais próxima** no
plano, e o resultado vai para um empacotamento de círculos: o círculo escuro é o grupo,
os claros dentro dele são as categorias que gravitam em torno daquele grupo.

Dois ajustes em relação ao script de referência:

* `explained_inertia_` **não existe** no `prince` 0.20 — o atributo virou
  `percentage_of_variance_`, já em percentual;
* o critério de corte passa a ser a **inércia acumulada nas duas dimensões**, e não o
  máximo de uma delas: um plano só é legível se as duas dimensões juntas retiverem o
  suficiente da tabela original. O piso é 80%.

In [18]:
#: Nominais com cardinalidade que sustenta uma tabela de contingência estável.
NOMINAIS_CA = ["bairro", "tipo_construcao", "estilo", "tipo_garagem"]

#: Inércia acumulada mínima nas duas dimensões para que o plano seja interpretável.
INERCIA_MINIMA = 80.0


def distancia_euclidiana(px, qx, py, qy):
    """Distância entre dois pontos do plano fatorial."""
    return float(np.sqrt((px - qx) ** 2 + (py - qy) ** 2))


def circulos_do_agrupamento(vinculos: pd.DataFrame) -> list:
    """Hierarquia total → grupo → categoria, no formato que o circlify espera."""
    dados = [
        {
            "id": "Total",
            "datum": len(vinculos),
            "children": [
                {
                    "id": grupo,
                    "datum": int((vinculos.grupo == grupo).sum()),
                    "children": [
                        {"id": categoria, "datum": 1}
                        for categoria in vinculos.query("grupo == @grupo").categoria
                    ],
                }
                for grupo in vinculos.grupo.unique()
            ],
        }
    ]
    return circlify.circlify(
        dados, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1)
    )


inercias = []

for p in range(len(NOMINAIS_CA)):
    for k in range(p + 1, len(NOMINAIS_CA)):
        primeira, segunda = NOMINAIS_CA[p], NOMINAIS_CA[k]

        # Tabela de contingência, sempre com a variável de maior cardinalidade nas linhas
        contingencia = pd.crosstab(df[primeira], df[segunda])
        if contingencia.shape[0] < contingencia.shape[1]:
            contingencia = contingencia.T
        if min(contingencia.shape) < 2:
            continue

        modelo = prince.CA(n_components=2, random_state=config.SEMENTE).fit(contingencia)
        inercia = float(np.sum(modelo.percentage_of_variance_))
        inercias.append(
            {
                "linhas": contingencia.index.name,
                "colunas": contingencia.columns.name,
                "inercia_2d": round(inercia, 2),
                "figura": inercia >= INERCIA_MINIMA,
            }
        )
        if inercia < INERCIA_MINIMA:
            continue

        # Cada categoria de linha vai para a categoria de coluna mais próxima no plano
        linhas = modelo.row_coordinates(contingencia)
        colunas = modelo.column_coordinates(contingencia)
        pares = []
        for i, categoria in enumerate(contingencia.index):
            distancias = [
                distancia_euclidiana(
                    linhas.iloc[i, 0], colunas.iloc[j, 0], linhas.iloc[i, 1], colunas.iloc[j, 1]
                )
                for j in range(len(colunas))
            ]
            pares.append(
                {
                    "categoria": categoria,
                    "grupo": contingencia.columns[int(np.argmin(distancias))],
                    "distancia": round(min(distancias), 3),
                }
            )
        vinculos = pd.DataFrame(pares)
        if vinculos.grupo.nunique() < 2:
            continue

        circulos = circulos_do_agrupamento(vinculos)

        fig, ax = plt.subplots(figsize=(11, 11))
        ax.axis("off")
        ax.set_aspect("equal")
        limite = max(max(abs(c.x) + c.r, abs(c.y) + c.r) for c in circulos)
        ax.set_xlim(-limite, limite)
        ax.set_ylim(-limite, limite)

        for circulo in circulos:  # o círculo de cada grupo
            if circulo.level == 2:
                ax.add_patch(plt.Circle((circulo.x, circulo.y), circulo.r, color=BARRA))

        for circulo in circulos:  # as categorias dentro de cada grupo
            if circulo.level == 3:
                ax.add_patch(
                    plt.Circle((circulo.x, circulo.y), circulo.r, alpha=0.55, color=DESTAQUE)
                )
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#12303F",
                )

        for circulo in circulos:  # as etiquetas dos grupos, por cima
            if circulo.level == 2:
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y + circulo.r * 0.82),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white",
                    bbox=dict(
                        facecolor=config.AZUL_PROFUNDO, edgecolor="none", boxstyle="round,pad=0.4"
                    ),
                )

        ax.set_title(
            f"{contingencia.index.name} agrupada por {contingencia.columns.name}"
            f" — inércia acumulada {inercia:.1f}%"
        )
        salvar(fig, f"multivariada-ca-{primeira} X {segunda}", vinculos)

inercias = pd.DataFrame(inercias).sort_values("inercia_2d", ascending=False)
io.salvar_tabela(inercias, SECAO, "multivariada-inercia-dos-pares", indice=False)
inercias

,linhas,colunas,inercia_2d,figura
4,tipo_garagem,tipo_construcao,96.68,True
0,bairro,tipo_construcao,88.54,True
2,bairro,tipo_garagem,88.17,True
5,estilo,tipo_garagem,86.75,True
1,bairro,estilo,76.36,False
3,estilo,tipo_construcao,71.67,False


# 5ª etapa · Detecção de valores ausentes

> *Detecção de valores ausentes é uma tarefa comum na área de ciência de dados para
> mensurar os impactos dessa ausência no conjunto de dados em estudo.*

A contagem roda sobre a **base completa**, com as 82 colunas — o recorte serve para
analisar, não para esconder buraco.

O ponto que muda a leitura: em Ames, quase toda a ausência é **estrutural**. `NA` em
`qualidade_piscina` não é dado perdido, é imóvel sem piscina. O projeto registra essas colunas em
`dic.AUSENCIA_ESTRUTURAL`, e o gráfico separa as duas naturezas por cor — tratar as duas
do mesmo jeito e imputar pela mediana inventaria piscinas que não existem.

In [19]:
ausentes = (
    bruto.isna()
    .sum()
    .rename("ausentes")
    .to_frame()
    .assign(percentual=lambda t: (100 * t.ausentes / len(bruto)).round(2))
    .query("ausentes > 0")
    .sort_values("ausentes", ascending=False)
    .reset_index(names="variavel")
)
ausentes["natureza"] = np.where(
    ausentes.variavel.isin(dic.AUSENCIA_ESTRUTURAL), "estrutural", "real"
)

fig, ax = plt.subplots(figsize=(15, 6))
sns.barplot(
    data=ausentes,
    x="variavel",
    y="percentual",
    hue="natureza",
    palette={"estrutural": DESTAQUE, "real": config.CORAL},
    ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.1f%%", fontsize=8, padding=2)
ax.tick_params(axis="x", rotation=60)
ax.set_xlabel("Características com ao menos uma célula vazia")
ax.set_ylabel("% de células ausentes")
ax.set_title(
    f"Ausência por coluna — {len(ausentes)} de {bruto.shape[1]} características afetadas"
)
ax.legend(title="Natureza da ausência")

salvar(fig, "valores-ausentes", ausentes)
ausentes

,variavel,ausentes,percentual,natureza
0,qualidade_piscina,2917,99.56,estrutural
1,item_extra,2824,96.38,estrutural
2,tipo_viela,2732,93.24,estrutural
3,tipo_cerca,2358,80.48,estrutural
4,qualidade_lareira,1422,48.53,estrutural
5,testada_lote,490,16.72,real
6,acabamento_garagem,159,5.43,estrutural
7,qualidade_garagem,159,5.43,estrutural
8,conservacao_garagem,159,5.43,estrutural
9,ano_garagem,159,5.43,real


# 6ª etapa · Detecção de valores discrepantes

> *A detecção de outliers consiste na identificação de padrões em dados com um
> comportamento diferente do esperado. Estes padrões são muitas vezes referidos como
> anomalias.*

O critério é o de Tukey — fora do intervalo `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]` —, já
implementado em `est.outliers_iqr`. A tabela do script de referência guardava apenas o
maior valor discrepante de cada variável; aqui vão também a contagem, o percentual e as
duas cercas, porque **o que decide se o registro sai da base é o volume**, não o extremo.

Discrepante não é erro. Um imóvel de 5.642 pés² existe; é a Meridiano que precisa decidir
se ele pertence à mesma carteira dos outros.

In [20]:
linhas = []
for column in numericas:
    serie = df[column].dropna()
    marca = est.outliers_iqr(serie)
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    linhas.append(
        {
            "variavel": column,
            "discrepantes": int(marca.sum()),
            "percentual": round(100 * marca.mean(), 2),
            "limite_inferior": q1 - 1.5 * iqr,
            "limite_superior": q3 + 1.5 * iqr,
            "menor_discrepante": serie[marca].min() if marca.any() else np.nan,
            "maior_discrepante": serie[marca].max() if marca.any() else np.nan,
        }
    )

discrepantes = pd.DataFrame(linhas).sort_values("percentual", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=discrepantes, x="percentual", y="variavel", color=BARRA, ax=ax)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f%%", fontsize=8, padding=3)
ax.axvline(discrepantes.percentual.mean(), color=REFERENCIA, ls="dashed", lw=2)
ax.set_xlabel("% de registros fora das cercas de Tukey")
ax.set_ylabel("")
ax.set_title("Discrepantes por característica — critério IQR (1,5×)")
ax.margins(x=0.15)

salvar(fig, "valores-discrepantes", discrepantes)
discrepantes

,variavel,discrepantes,percentual,limite_inferior,limite_superior,menor_discrepante,maior_discrepante
5,conservacao_geral,252,8.61,3.500,7.500,1.0,9.0
0,testada_lote,187,7.67,25.000,113.000,21.0,313.0
10,preco_venda,137,4.68,3500.000,339500.000,339750.0,755000.0
3,area_terreno,127,4.34,1267.875,17728.875,17755.0,215245.0
1,area_porao,122,4.17,29.500,2065.500,0.0,6110.0
7,dormitorios,78,2.66,0.500,4.500,0.0,8.0
6,area_habitavel,74,2.53,200.500,2668.500,2673.0,5642.0
8,comodos,51,1.74,2.000,10.000,11.0,15.0
2,area_garagem,42,1.43,-64.000,960.000,962.0,1488.0
9,lareiras,13,0.44,-1.500,2.500,3.0,4.0


# 7ª etapa · Detecção de correlação

> *Em probabilidade e estatística, correlação, dependência ou associação é qualquer
> relação estatística entre duas ou mais variáveis.*

O script lê a matriz `auto` do `ProfileReport`, que escolhe a medida conforme o par de
tipos. Sem a biblioteca, a mesma matriz é montada aqui com as três medidas que o par
exige — todas já implementadas em `src/eda/estatistica.py`:

| Par de tipos | Medida | Por quê |
|:---|:---|:---|
| numérica × numérica | Spearman | monótona, não presume linearidade nem normalidade |
| nominal × nominal | V de Cramér | qui-quadrado normalizado, com correção de Bergsma |
| nominal × numérica | η (razão de correlação) | fração da variância que a categoria explica |

As ordinais entram como numéricas **depois de codificadas pela escala declarada** — é o
que dá sentido a medir monotonicidade nelas. Todas as três medidas vivem em `[0, 1]`, o
que torna a matriz comparável célula a célula.

In [21]:
# As ordinais viram números pela escala declarada; ausência estrutural vale 0.
codificado = df.copy()
for column in ordinal_columns:
    codificado[column] = codificado[column].map(dic.ESCALAS_ORDINAIS[column]).fillna(0)

quantitativas = numericas + ordinal_columns
qualitativas = nominal_columns
colunas = quantitativas + qualitativas

correlacao = pd.DataFrame(np.eye(len(colunas)), index=colunas, columns=colunas)
for i, primeira in enumerate(colunas):
    for segunda in colunas[i + 1 :]:
        if primeira in quantitativas and segunda in quantitativas:
            valor = codificado[primeira].corr(codificado[segunda], method="spearman")
        elif primeira in qualitativas and segunda in qualitativas:
            valor = est.v_de_cramer(codificado[primeira], codificado[segunda])
        else:
            categorica, quantitativa = (
                (primeira, segunda) if primeira in qualitativas else (segunda, primeira)
            )
            valor = np.sqrt(est.razao_correlacao(codificado[categorica], codificado[quantitativa]))
        correlacao.loc[primeira, segunda] = correlacao.loc[segunda, primeira] = abs(float(valor))

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    correlacao,
    cmap="Blues",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6},
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("Associação entre características — Spearman, V de Cramér e η")

salvar(fig, "correlacao", correlacao.reset_index(names="variavel"))

# Pares acima do limiar de redundância: a mesma informação medida duas vezes.
redundantes = est.pares_redundantes(correlacao, limiar=0.7)
io.salvar_tabela(redundantes, SECAO, "pares-redundantes", indice=False)
redundantes

,variavel_a,variavel_b,correlacao
0,qualidade_geral,preco_venda,0.809
1,area_habitavel,comodos,0.808
2,preco_venda,bairro,0.757
3,qualidade_geral,bairro,0.739
4,qualidade_externa,bairro,0.734
5,qualidade_externa,qualidade_cozinha,0.724
6,area_habitavel,preco_venda,0.724
7,qualidade_geral,qualidade_externa,0.720


# 8ª etapa · Seleção de características

> *Os métodos de seleção de características têm a missão de encontrar uma combinação
> adequada de características relevantes para o aprendizado de um conceito.*

O script de referência encerra registrando o tamanho do conjunto de entrada. Aqui o
registro vira **decisão**, com o motivo ao lado de cada variável — é o que a fase de
preparação recebe como entrada:

* **reservada** — `preco_venda` é o rótulo da avaliação a posteriori e `bairro` é
  proxy de preço, guardada para validação externa. Nenhuma das duas entra na segmentação,
  sob pena de o modelo redescobrir o preço e chamar isso de padrão;
* **descartada por redundância** — dentro de cada par acima de 0,7 sai a medida de
  **menor resolução**, isto é, a que distingue menos valores: entre `area_habitavel`
  (1.292 valores distintos) e `comodos` (14), as duas dizem "tamanho", mas só uma
  diz quanto. Empate de cardinalidade se resolve pela maior associação média com o
  resto. O critério **não olha para `preco_venda`**: usar o rótulo reservado para escolher
  característica seria trazê-lo de volta para dentro da modelagem pela porta dos fundos;
* **mantida** — o resto.

In [22]:
# Conjunto de dados inicial. O script de referência troca os rótulos de "numéricas" e
# "categóricas" ao montar este quadro; aqui cada contagem fica sob o próprio nome.
df_inicial = pd.DataFrame(
    [
        {
            "Quantidade de características": df_inicial_qnt_feature,
            "Características numéricas": df_inicial_qnt_feature_Numeric,
            "Características categóricas": df_inicial_qnt_feature_Categorical,
        }
    ]
)
io.salvar_tabela(df_inicial, SECAO, "df_inicial", indice=False)

RESERVADAS = {
    "preco_venda": "rótulo reservado para a avaliação a posteriori",
    "bairro": "proxy de preço, reservada para validação externa",
}

decisoes = {
    variavel: ("reservada", motivo) for variavel, motivo in RESERVADAS.items()
}

# Dentro de cada par redundante fica a medida de maior resolução; o desempate é a
# associação média com o resto do conjunto.
for _, par in redundantes.iterrows():
    a, b = par.variavel_a, par.variavel_b
    if a in decisoes or b in decisoes:
        continue
    if df[a].nunique() != df[b].nunique():
        descartada, mantida = (a, b) if df[a].nunique() < df[b].nunique() else (b, a)
        criterio = f"{df[descartada].nunique()} valores distintos contra {df[mantida].nunique()}"
    else:
        ativas = [c for c in correlacao.columns if c not in decisoes]
        media_a = correlacao.loc[a, ativas].drop(a).mean()
        media_b = correlacao.loc[b, ativas].drop(b).mean()
        descartada, mantida = (a, b) if media_a > media_b else (b, a)
        criterio = "mesma cardinalidade; sai a de maior associação média"
    decisoes[descartada] = (
        "descartada",
        f"associação {par.correlacao:.2f} com {mantida} · {criterio}",
    )

tipos = dicionario.set_index("variavel").Tipo
selecao = pd.DataFrame(
    [
        {
            "variavel": variavel,
            "tipo": tipos.get(variavel, "—"),
            "decisao": decisoes.get(variavel, ("mantida", "sem redundância acima do limiar"))[0],
            "motivo": decisoes.get(variavel, ("mantida", "sem redundância acima do limiar"))[1],
        }
        for variavel in correlacao.columns
    ]
).sort_values(["decisao", "variavel"])

io.salvar_tabela(selecao, SECAO, "selecao-de-caracteristicas", indice=False)

print(
    f"{(selecao.decisao == 'mantida').sum()} mantidas · "
    f"{(selecao.decisao == 'descartada').sum()} descartadas por redundância · "
    f"{(selecao.decisao == 'reservada').sum()} reservadas"
)
selecao

19 mantidas · 2 descartadas por redundância · 2 reservadas


,variavel,tipo,decisao,motivo
8,comodos,Discreta,descartada,associação 0.81 com area_habitavel · 14 valore...
11,qualidade_externa,Ordinal,descartada,associação 0.72 com qualidade_cozinha · 4 valo...
14,acabamento_garagem,Ordinal,mantida,sem redundância acima do limiar
19,ar_central,Nominal,mantida,sem redundância acima do limiar
2,area_garagem,Contínua,mantida,sem redundância acima do limiar
6,area_habitavel,Discreta,mantida,sem redundância acima do limiar
1,area_porao,Contínua,mantida,sem redundância acima do limiar
3,area_terreno,Discreta,mantida,sem redundância acima do limiar
22,condicao_venda,Nominal,mantida,sem redundância acima do limiar
5,conservacao_geral,Discreta,mantida,sem redundância acima do limiar


# Insights

O que as oito etapas deixaram, com o número que sustenta cada afirmação. É esta lista
que a fase de preparação consome — cada item ou vira uma decisão de tratamento, ou vira
uma restrição que a modelagem precisa respeitar.

In [19]:
insights = [
    "Duas linhas são idênticas nas 24 características analisadas e só diferem em "
    "`ordem` e `pid`: o mesmo imóvel contado duas vezes em qualquer média.",
    "Das 27 colunas com célula vazia, 14 são ausência estrutural — `qualidade_piscina` 99,6%, "
    "`tipo_viela` 93,2%, imóvel sem piscina e sem viela. `testada_lote` (16,7%) é a única "
    "ausência de fato desconhecida com peso; o resto não passa de 0,8%.",
    "`area_terreno` tem assimetria 12,8, a maior da base: mediana de 9.437 pés² contra "
    "máximo de 215.245 — transformação obrigatória antes de qualquer distância.",
    "`conservacao_geral` concentra 56% dos imóveis na nota 5, e por isso 8,6% dos registros "
    "caem fora das cercas de Tukey: é discrepância de escala, não imóvel atípico.",
    "A qualidade explica o preço melhor que a área: `qualidade_geral` × `preco_venda` = 0,81 "
    "em Spearman, contra 0,72 de `area_habitavel`.",
    "`area_habitavel` e `comodos` medem a mesma coisa (0,81): tamanho contado em pés "
    "quadrados e em cômodos, redundância a resolver na preparação.",
    "O bairro sozinho responde por η = 0,76 do preço — Northridge vende a 330 mil, 183% da "
    "média da carteira. É o motivo de `bairro` ficar reservada para validação.",
    "Ar central está em 93% dos imóveis, mas os 7% sem ele valem 102 mil contra 186 mil: "
    "uma variável binária que separa metade do valor da carteira.",
    "A venda é sazonal: junho e julho concentram 33% das transações, janeiro e dezembro "
    "somam 8% — comparar volumes sem corrigir o mês compara estações do ano.",
    "2010 tem 341 registros e termina em julho: a série está truncada, e qualquer "
    "leitura de queda no último ano é artefato de coleta.",
    "82% das vendas são `Normal`; `Parcial` (8,4%) é imóvel entregue antes da conclusão "
    "do cadastro e carrega preço acima da média por construção, não por qualidade.",
    "`zoneamento` e `tipo_garagem` compartilham 99% da inércia em duas dimensões: o "
    "zoneamento praticamente determina o tipo de garagem do imóvel.",
]

insights = pd.DataFrame(insights, columns=["insights"])
io.salvar_tabela(insights, SECAO, "insights", indice=False)

for numero, texto in enumerate(insights.insights, start=1):
    print(f"{numero:>2}. {texto}\n")

 1. Duas linhas são idênticas nas 24 características analisadas e só diferem em `Order` e `PID`: o mesmo imóvel contado duas vezes em qualquer média.

 2. Das 27 colunas com célula vazia, 14 são ausência estrutural — `Pool QC` 99,6%, `Alley` 93,2%, imóvel sem piscina e sem viela. `Lot Frontage` (16,7%) é a única ausência de fato desconhecida com peso; o resto não passa de 0,8%.

 3. `Lot Area` tem assimetria 12,8, a maior da base: mediana de 9.437 pés² contra máximo de 215.245 — transformação obrigatória antes de qualquer distância.

 4. `Overall Cond` concentra 56% dos imóveis na nota 5, e por isso 8,6% dos registros caem fora das cercas de Tukey: é discrepância de escala, não imóvel atípico.

 5. A qualidade explica o preço melhor que a área: `Overall Qual` × `SalePrice` = 0,81 em Spearman, contra 0,72 de `Gr Liv Area`.

 6. `Gr Liv Area` e `TotRms AbvGrd` medem a mesma coisa (0,81): tamanho contado em pés quadrados e em cômodos, redundância a resolver na preparação.

 7. O bairro so

## Próximo passo

As oito etapas deste roteiro descrevem a base **como ela chega**. O notebook
[`02-eda-fluxo-8-fases`](02-eda-fluxo-8-fases.ipynb) percorre o mesmo fluxo sobre a
**base curada**, com as variáveis derivadas, as escalas já codificadas e a ausência
estrutural resolvida — é lá que as decisões listadas na 8ª etapa aparecem aplicadas.

As figuras deste notebook estão em `docs/imagens/eda/` e as tabelas em
`reports/tables/eda-exploratoria/`.